# 05. Does an automated annotator agree with my markers, and where it does not, who is right?

CellTypist is a classifier trained on immune cells from many tissues. You run it on the same cells and compare it with your manual labels and the authors'. Three annotators, one truth.

**First move, in the terminal:** `uv add celltypist`

**Input:** `ra_pbmc_annotated.h5ad` · **Output:** `C:/bio_data/ra_pbmc_celltypist.h5ad`

> **How this notebook works.** Each step has a **spec** (what the code must do), a **read-first** link, an empty code cell for you, and a **TEST** cell written by Claude. Your cell is done when its test prints PASS and you can say why your approach works. The ✍️ cells are yours to write; predictions go in *before* you run the cell they predict.
>
> No function names are given on purpose. Find them in the read-first link or the scanpy docs. Stuck for more than 10 minutes on one cell: ask Claude for a *question*, not code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import celltypist

In [ ]:
# adata = the annotated file from 04

...

## T1. What does the model expect?

**Spec:** download the `Immune_All_Low.pkl` model and load it. Print how many cell types it knows and how many genes it uses. Then check that your input is what CellTypist requires: log1p of counts scaled to 10,000 per cell, **all genes** (not only the HVGs).

**Read first:** https://www.celltypist.org/tutorials and the input requirements in https://github.com/Teichlab/celltypist (README, section on the input data).

In [ ]:
# download + load the model
# print number of cell types and genes
# check your X meets the input requirement

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
s = np.asarray(np.expm1(adata.X[:1000]).sum(axis=1)).ravel()
assert np.allclose(s, 1e4, rtol=0.01), "CellTypist wants log1p of counts scaled to 10,000"
assert adata.n_vars > 10_000, "use all genes, not the HVG subset"
print("PASS: input fits the model")

✍️ **What share of the model's genes are in your data? What happens to a model when a gene it relies on is missing?**

_..._

## T2. Predict

**Spec:** run CellTypist with majority voting over **your** Leiden clusters. Store `predicted_labels`, `majority_voting` and `conf_score` in `adata.obs` as `ct_label`, `ct_majority`, `ct_conf`.

✍️ **Prediction: will CellTypist find more types than you, or fewer? Which of yours will it split?**

_..._

In [ ]:
# annotate with majority voting over obs['leiden']
# obs['ct_label'], obs['ct_majority'], obs['ct_conf']

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
for col in ["ct_label", "ct_majority", "ct_conf"]:
    assert col in adata.obs, col
assert adata.obs["ct_conf"].between(0, 1).all()
print("PASS:", adata.obs["ct_label"].nunique(), "CellTypist labels")

## T3. How sure is it?

**Spec:** histogram of `ct_conf`, and the median confidence per `ct_label` (sorted). Store the share of cells under 0.5 confidence in `share_unsure`.

In [ ]:
# histogram + median confidence per label
# share_unsure

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert 0 <= share_unsure <= 1
print(f"PASS: {100 * share_unsure:.1f}% of cells under 0.5 confidence")

✍️ **Which labels is it least sure about? Do they overlap with the clusters you found unclear in 04 A2?**

_..._

## T4. Three-way comparison

**Spec:** a dict `ct_to_rough` mapping every CellTypist majority label to the authors' five rough classes, then `agreement_ct` (share where it matches `rough_annot`). Then a crosstab `my_label` vs `ct_majority`, normalized per row, shown as a heatmap.

In [ ]:
# ct_to_rough, obs['ct_rough'], agreement_ct
# heatmap my_label vs ct_majority

...

In [ ]:
# TEST: do not edit. Run it after your cell; it must print PASS.
assert set(adata.obs["ct_majority"]) <= set(ct_to_rough), "not every CellTypist label is mapped"
assert 0 <= agreement_ct <= 1
print(f"PASS: CellTypist {100 * agreement_ct:.1f}% vs you {100 * (adata.obs['my_rough'] == adata.obs['rough_annot']).mean():.1f}% at the rough level")

✍️ **The headline of Artifact 2 is here: where do your markers and CellTypist disagree? Pick the 3 largest disagreements.**

_..._

## T5. Settle the disagreements

**Spec:** for each of your 3 disagreements, one dotplot of the markers that tell the two candidate types apart, on only those cells.

In [ ]:
# 3 dotplots, one per disagreement

...

✍️ **For each one: who was right (you, CellTypist, or the authors), and the evidence. One of these is your README finding.**

_..._

## T6. Save

**Spec:** write to `C:/bio_data/ra_pbmc_celltypist.h5ad`.

In [ ]:
# write the file

...

## Teach-back

3 to 5 sentences in your own words: what this notebook did, what it found, and one thing that surprised you. No sentence means not learned.

_..._

## AI note

What did Claude write (the specs, the tests), what did you write, and where did you get stuck and ask for a hint?

_..._